# Multi-step CreativeIR v0.1 decompilation

Two-pass **Gemini 3.8 Flash** decompilation (via **OpenRouter**, native video input) with deterministic
perception overrides: ffprobe media facts + PySceneDetect shot boundaries are authoritative,
Gemini provides the semantic shot analysis and global creative synthesis.

Set `OPENROUTER_API_KEY` before running (env var or `.env` at the repo root).
`OPENROUTER_MODEL` overrides the default `google/gemini-3.8-flash`.


In [ ]:
%pip install -q -U requests jsonschema scenedetect

import copy
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, display
from jsonschema import Draft202012Validator

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "scripts"))
from perceive import probe_media, detect_scenes
from openrouter_client import call_openrouter_json

video_id = os.environ.get("OPENROUTER_VIDEO_ID", "7106594312292453675")
default_source_dir = repo_root / "data" / "exploration" / video_id
source_dir = Path(os.environ.get("OPENROUTER_SOURCE_DIR", default_source_dir)).expanduser().resolve()
fallback_source = repo_root.parent / "tiktok-factory" / ".orca" / "drops"
if not source_dir.exists() and fallback_source.exists():
    source_dir = fallback_source
video_path = source_dir / "video.mp4"
metadata_path = source_dir / "metadata.json"
schema_path = repo_root / "schemas" / "creative_ir_v0_1.json"
raw_shot_path = source_dir / "creative_ir.shot_analysis.raw.json"
raw_synth_path = source_dir / "creative_ir.global_synth.raw.json"
parsed_path = source_dir / "creative_ir.parsed.json"
baseline_3_path = source_dir / "creative_ir.baseline_3.json"
usage_path = source_dir / "creative_ir.usage.json"
perception_path = source_dir / "perception.json"
note_path = source_dir / "creative_ir.implementation.md"
for path in (video_path, metadata_path, schema_path):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
schema = json.loads(schema_path.read_text(encoding="utf-8"))
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
source_dir.mkdir(parents=True, exist_ok=True)

# Save #3 baseline before it gets overwritten by the new run
if parsed_path.exists() and not baseline_3_path.exists():
    import shutil
    shutil.copy2(parsed_path, baseline_3_path)
    print(f"Saved #3 baseline: {baseline_3_path}")

print(f"video={video_path}")
print(f"schema={schema_path}")
print(f"source_dir={source_dir}")


## Step 1: Deterministic perception

ffprobe extracts exact media facts (duration, resolution, fps, codecs). PySceneDetect detects hard shot boundaries. These values are authoritative and will be injected into the Gemini context rather than guessed.

In [ ]:
# Probe media facts
media = probe_media(video_path)
media_dict = media.to_dict()
print("Media facts (from ffprobe):")
print(json.dumps(media_dict, indent=2))

# Detect scenes
scenes = detect_scenes(video_path)
print(f"\nDetected {len(scenes)} scenes:")
for i, s in enumerate(scenes):
    print(f"  scene_{i}: {s['start_seconds']:.3f}s -> {s['end_seconds']:.3f}s ({s['end_seconds'] - s['start_seconds']:.2f}s)")

# Extract representative frames (one per scene) for Gemini context
frame_dir = source_dir / "frames"
frame_dir.mkdir(exist_ok=True)
frame_paths = []
for i, s in enumerate(scenes):
    mid = (s["start_seconds"] + s["end_seconds"]) / 2
    frame_path = frame_dir / f"scene_{i:03d}.jpg"
    subprocess.run([
        "ffmpeg", "-y", "-ss", str(mid), "-i", str(video_path),
        "-frames:v", "1", "-q:v", "2", str(frame_path),
    ], capture_output=True, check=True)
    frame_paths.append(frame_path)
    print(f"  Extracted frame: {frame_path.name}")

# Persist perception
perception = {
    "media_facts": media_dict,
    "scenes": scenes,
    "frame_paths": [str(p.relative_to(source_dir)) for p in frame_paths],
    "probe_raw": {"streams": media.raw_streams, "format": media.raw_format},
}
perception_path.write_text(json.dumps(perception, indent=2) + "\n", encoding="utf-8")
print(f"\nSaved perception: {perception_path}")

## Step 2: Shot analysis (Gemini 3.8 Flash via OpenRouter)

Gemini receives the full video (inline base64 via OpenRouter `video_url`) plus deterministic
scene boundaries and media facts, and returns a full CreativeIR with per-shot analysis.


In [ ]:
# The full repository schema is embedded in the prompts; validate it here so
# any schema edit breaks loudly before model calls are made.
Draft202012Validator.check_schema(schema)
print("CreativeIR v0.1 schema valid:", list(schema["properties"]))


In [ ]:
model_name = os.environ.get("OPENROUTER_MODEL", "google/gemini-3.8-flash")
shot_prompt_version = "openrouter-gemini-shot-analysis-v0.1"
metadata_context = json.dumps({key: metadata.get(key) for key in ("source_url", "video_id", "creator", "caption", "hashtags", "publication_date")}, ensure_ascii=False)

# Build scene boundary context for the prompt
scene_context = json.dumps(scenes, indent=2)
media_context = json.dumps(media_dict, indent=2)

shot_prompt = f"""You are a meticulous audiovisual decompiler performing shot-level analysis.

DETERMINISTIC FACTS (from ffprobe, do NOT override):
{media_context}

DETECTED SHOT BOUNDARIES (from PySceneDetect, do NOT override timestamps):
{scene_context}

SOURCE METADATA:
{metadata_context}

Analyze each detected shot in the video. For every shot provide:
- Visual: exact description, subjects, environment, palette
- Camera: framing, angle, motion, composition
- Text: every legible on-screen text segment with exact OCR, timing, placement, role
- Dialogue: presence (present/absent/uncertain), exact words if present
- Audio: music/original sound presence and label, sound effects, mix notes. Keep audio descriptions CONSERVATIVE — do not invent specific sound identities, song names, or effects you cannot confirm. Use identity_known=false unless you can identify the audio source.
- Editing: transition_in, transition_out, pacing, notes
- Evidence: at least one evidence entry per shot referencing the video time range

Also provide:
- Per-shot semantic_role, attention_mechanisms, confidence, rationale
- Per-shot reconstruction_prompt and continuity_requirements

The complete repository CreativeIR v0.1 schema is authoritative for every nested field:
{json.dumps(schema, ensure_ascii=False, separators=(",", ":"))}

CRITICAL RULES:
- Use the DETECTED shot boundaries above exactly — do not invent new boundaries
- Use the DETECTED media facts exactly — do not guess duration, resolution, or fps
- Record exact visible OCR text. If you see text that is readable, transcribe it faithfully.
- Keep uncertain audio/OCR claims explicitly uncertain
- Do not copy caption text into on-screen OCR unless those exact words are visibly rendered
- Ensure decompilation.model is exactly {model_name!r}, prompt_version is {shot_prompt_version!r}, schema_version is "0.1", annotator_type is "automated"
- Return ONLY the CreativeIR JSON object matching the repository schema
"""

shot_result, shot_call, shot_repair = call_openrouter_json(shot_prompt, model=model_name, video_path=video_path)
raw_shot_path.write_text(shot_call.content, encoding="utf-8")
print(f"Shot analysis raw response saved: {raw_shot_path}")
if shot_repair is not None:
    (source_dir / "creative_ir.shot_analysis.repaired.json").write_text(shot_repair.content, encoding="utf-8")
    print("Shot analysis JSON was repaired by a follow-up pass (original preserved)")
print(f"Model: {shot_call.model} (id={shot_call.response_id}, finish={shot_call.finish_reason})")
usage_shot = dict(shot_call.usage)
if shot_repair is not None:
    for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
        usage_shot[key] = (usage_shot.get(key) or 0) + (shot_repair.usage.get(key) or 0)
    if isinstance(usage_shot.get("cost"), (int, float)) and isinstance(shot_repair.usage.get("cost"), (int, float)):
        usage_shot["cost"] = usage_shot["cost"] + shot_repair.usage["cost"]
    usage_shot["repair_calls"] = 1
print("Usage:", json.dumps({k: usage_shot.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens", "cost")}, indent=2))


## Step 3: Inject deterministic facts

Override the Gemini-provided source media facts and shot timestamps with the exact deterministic values from ffprobe and PySceneDetect. This ensures accuracy without relying on the model.

In [ ]:
def _parse_shot_index(shot_id: str) -> int | None:
    """Safely extract numeric index from shot_N id; returns None if malformed."""
    try:
        return int(shot_id.split("_")[1])
    except (IndexError, ValueError):
        return None

def _shot_span(shot_ids: list[str], scenes: list[dict]) -> dict | None:
    """Time range spanning the referenced shots, robust to unordered ids."""
    indices = [_parse_shot_index(s) for s in shot_ids]
    indices = [i for i in indices if i is not None and 0 <= i < len(scenes)]
    if not indices:
        return None
    first_idx, last_idx = min(indices), max(indices)
    return {
        "start_seconds": scenes[first_idx]["start_seconds"],
        "end_seconds": scenes[last_idx]["end_seconds"],
    }

def inject_deterministic_facts(ir: dict, media: 'PerceptionResult', scenes: list[dict]) -> dict:
    """Override model-guessed values with authoritative deterministic facts."""
    ir = copy.deepcopy(ir)

    # Source media facts
    src = ir.setdefault("source", {}).setdefault("observed", {})
    src["duration_seconds"] = media.duration_seconds
    src["frame_size"] = {"width": media.width, "height": media.height}
    src["aspect_ratio"] = media.aspect_ratio_label
    src.setdefault("evidence", []).append({
        "kind": "metadata",
        "note": f"Exact values from ffprobe (duration={media.duration_seconds:.3f}s, {media.width}x{media.height}, fps={media.fps:.2f}, codec={media.video_codec})"
    })

    # Shot time ranges from detected scenes: every scene must map to exactly one shot
    shots = ir.get("observed", {}).get("shots", [])
    if len(shots) != len(scenes):
        raise ValueError(
            f"Shot count mismatch: model returned {len(shots)} shots but PySceneDetect detected {len(scenes)}. "
            "Re-run the shot analysis pass before continuing."
        )
    for i, shot in enumerate(shots):
        shot["time_range"] = {
            "start_seconds": scenes[i]["start_seconds"],
            "end_seconds": scenes[i]["end_seconds"],
        }
        shot.setdefault("observed", {}).setdefault("evidence", []).append({
            "kind": "timing",
            "note": f"Exact boundary from PySceneDetect scene {i}"
        })

    # Update narrative beat time ranges if they reference shots
    beats = ir.get("observed", {}).get("narrative", {}).get("beats", [])
    for beat in beats:
        beat_shots = beat.get("shot_ids", [])
        if beat_shots:
            span = _shot_span(beat_shots, scenes)
            if span:
                beat["time_range"] = span

    # Update hook time range (append timing evidence, keep model evidence)
    hook = ir.get("observed", {}).get("hook", {})
    hook_shots = hook.get("shot_ids", [])
    if hook_shots:
        span = _shot_span(hook_shots, scenes)
        if span:
            hook.setdefault("evidence", []).append({
                "kind": "timing",
                "note": "Hook span recomputed from PySceneDetect boundaries",
                "time_range": span,
            })

    return ir

shot_injected = inject_deterministic_facts(shot_result, media, scenes)
print(f"Injected deterministic facts: duration={media.duration_seconds:.3f}s, {len(scenes)} shots")
print(f"Shot boundaries: {[(s['shot_id'], s['time_range']) for s in shot_injected['observed']['shots']]}")


## Step 4: Global creative synthesis (Gemini 3.8 Flash via OpenRouter)

A second call receives the deterministic-fact-injected shot analysis plus the original video. It infers hook, narrative arc, audience, attention/marketing mechanisms, and produces detailed model-agnostic reconstruction instructions including timeline, shot duration, composition, text treatment, transitions, pacing, continuity and payoff timing.


In [ ]:
synth_prompt_version = "openrouter-gemini-global-synthesis-v0.1"

# Build shot summary for the synthesis prompt
shot_summaries = []
for shot in shot_injected.get("observed", {}).get("shots", []):
    shot_summaries.append({
        "shot_id": shot["shot_id"],
        "time_range": shot["time_range"],
        "visual": shot.get("observed", {}).get("visual", {}).get("description", ""),
        "subjects": shot.get("observed", {}).get("visual", {}).get("subjects", []),
        "text_segments": [
            {"text": seg["text"], "role": seg["role"], "placement": seg["placement"]}
            for seg in shot.get("observed", {}).get("text", {}).get("segments", [])
        ],
        "dialogue_presence": shot.get("observed", {}).get("dialogue", {}).get("presence", "unknown"),
        "audio_label": shot.get("observed", {}).get("audio", {}).get("music_or_original_sound", {}).get("label", ""),
        "semantic_role": shot.get("inferred", {}).get("semantic_role", "other"),
    })

synth_prompt = f"""You are a creative strategist performing global synthesis of a decompiled video.

DETERMINISTIC MEDIA FACTS:
{json.dumps(media_dict, indent=2)}

DETECTED SHOT STRUCTURE ({len(scenes)} shots):
{json.dumps(scenes, indent=2)}

SHOT-LEVEL ANALYSIS (from prior Gemini pass):
{json.dumps(shot_summaries, indent=2)}

SOURCE METADATA:
{metadata_context}

Based on the above factual shot analysis and the original video, produce the GLOBAL sections of the CreativeIR:
1. observed.context (visible_subject, caption_signal, evidence)
2. observed.hook (shot_ids, text_ids, visual_summary, evidence with time ranges)
3. observed.narrative (beats with beat_id, label, shot_ids, time_range, visible_event)
4. observed.marketing (call_to_action_text_ids, engagement_devices, evidence)
5. observed.commercial (product_presence, evidence)
6. inferred.overall_concept (premise, format, viewer_action, confidence, rationale)
7. inferred.target_audience (primary_audience, interest_clusters, confidence, rationale)
8. inferred.hook (hook_types, promise, confidence, rationale)
9. inferred.narrative (story_summary, arc, payoff, confidence, rationale)
10. inferred.marketing (mechanisms, confidence, rationale)
11. inferred.commercial (status, problem, desire, promise, offer, proof_type, trust_signals, objections_addressed, cta_type, confidence, rationale)
12. generation.global_reconstruction_brief (detailed: timeline, shot duration, composition, text treatment, transitions, pacing, continuity, payoff timing)
13. generation.shot_order
14. generation.global_constraints

CRITICAL:
- Use the exact shot IDs and time ranges from the shot analysis above
- The reconstruction brief must be detailed enough to reproduce the creative without seeing the original
- Include shot-level timing, composition details, text styling, transitions, and pacing notes
- Keep audio descriptions conservative
- Return ONLY a JSON object with these global fields (not the shots themselves — those are already provided)
- Ensure decompilation.model is exactly {model_name!r}, prompt_version is {synth_prompt_version!r}

The complete repository CreativeIR v0.1 schema:
{json.dumps(schema, ensure_ascii=False, separators=(",", ":"))}
"""

synth_result, synth_call, synth_repair = call_openrouter_json(synth_prompt, model=model_name, video_path=video_path)
raw_synth_path.write_text(synth_call.content, encoding="utf-8")
print(f"Global synthesis raw response saved: {raw_synth_path}")
if synth_repair is not None:
    (source_dir / "creative_ir.global_synth.repaired.json").write_text(synth_repair.content, encoding="utf-8")
    print("Global synthesis JSON was repaired by a follow-up pass (original preserved)")
usage_synth = dict(synth_call.usage)
if synth_repair is not None:
    for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
        usage_synth[key] = (usage_synth.get(key) or 0) + (synth_repair.usage.get(key) or 0)
    if isinstance(usage_synth.get("cost"), (int, float)) and isinstance(synth_repair.usage.get("cost"), (int, float)):
        usage_synth["cost"] = usage_synth["cost"] + synth_repair.usage["cost"]
    usage_synth["repair_calls"] = 1
print("Usage:", json.dumps({k: usage_synth.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens", "cost")}, indent=2))


## Step 5: Merge and validate

Combine the shot-level analysis with the global synthesis into one complete CreativeIR v0.1 object. Inject deterministic facts. Validate against the repository schema and temporal integrity checks.

In [ ]:
def merge_creative_ir(shot_ir: dict, synth_global: dict) -> dict:
    """Merge shot analysis and global synthesis into one CreativeIR."""
    merged = copy.deepcopy(shot_ir)

    # Replace global sections from synthesis
    for section in ("context", "hook", "narrative", "marketing", "commercial"):
        if section in synth_global.get("observed", {}):
            merged.setdefault("observed", {})[section] = synth_global["observed"][section]

    # Replace inferred sections
    if "inferred" in synth_global:
        merged["inferred"] = synth_global["inferred"]

    # Replace generation sections
    if "generation" in synth_global:
        merged["generation"] = synth_global["generation"]

    # Ensure version
    merged["creative_ir_version"] = "0.1"

    return merged


merged = merge_creative_ir(shot_injected, synth_result)

# shot_injected already contains deterministic facts from Step 3;
# merge preserves shots, so no second injection needed (avoids duplicate evidence).
final = merged

# Set decompilation block
final["decompilation"] = {
    "model": model_name,
    "prompt_version": f"{shot_prompt_version}+{synth_prompt_version}",
    "schema_version": "0.1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "pipeline_version": "issue-4-multistep-perception-v0.1",
    "annotator_type": "automated",
}

# Validate against schema
Draft202012Validator(schema).validate(final)

# Temporal integrity check
def assert_temporal_integrity(ir):
    duration = ir["source"]["observed"]["duration_seconds"]
    shots = ir["observed"]["shots"]
    assert shots, "at least one shot is required"
    previous_end = 0.0
    for shot in shots:
        start = shot["time_range"]["start_seconds"]
        end = shot["time_range"]["end_seconds"]
        assert 0 <= start < end <= duration + 0.05, (shot["shot_id"], start, end, duration)
        assert start >= previous_end - 0.05, "shot ranges must be ordered"
        previous_end = end
    assert abs(shots[0]["time_range"]["start_seconds"]) <= 0.05
    assert abs(shots[-1]["time_range"]["end_seconds"] - duration) <= 0.05
    assert ir["generation"]["shot_order"] == [shot["shot_id"] for shot in shots]

assert_temporal_integrity(final)
parsed_path.write_text(json.dumps(final, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Draft 2020-12 validation and temporal/reference checks passed.")
print(f"Saved: {parsed_path}")
print(f"Shots: {len(final['observed']['shots'])}")
print(f"Duration: {final['source']['observed']['duration_seconds']:.3f}s")

## Usage and cost record

In [ ]:
def _usage_summary(usage: dict) -> dict:
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "cost_usd": usage.get("cost"),
    }

shot_usage_summary = _usage_summary(usage_shot or {})
synth_usage_summary = _usage_summary(usage_synth or {})
costs = [u["cost_usd"] for u in (shot_usage_summary, synth_usage_summary) if isinstance(u.get("cost_usd"), (int, float))]

usage_record = {
    "provider": "openrouter",
    "model": model_name,
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "video_id": video_id,
    "calls": [
        {"pass": "shot_analysis", "prompt_version": shot_prompt_version, "response_id": shot_call.response_id, "finish_reason": shot_call.finish_reason, "repairs": (usage_shot or {}).get("repair_calls", 0), "usage": shot_usage_summary},
        {"pass": "global_synthesis", "prompt_version": synth_prompt_version, "response_id": synth_call.response_id, "finish_reason": synth_call.finish_reason, "repairs": (usage_synth or {}).get("repair_calls", 0), "usage": synth_usage_summary},
    ],
    "cost_usd_total": round(sum(costs), 6) if costs else None,
}
usage_path.write_text(json.dumps(usage_record, indent=2) + "\n", encoding="utf-8")
print(json.dumps(usage_record, indent=2))


## Step 6: Compare with #3 baseline

In [ ]:
# Read baseline that was saved before this run overwrote it
if baseline_3_path.exists():
    baseline = json.loads(baseline_3_path.read_text(encoding="utf-8"))
    print("=== COMPARISON: #3 Baseline vs Multistep ===")
    print(f"\nBaseline: {baseline['decompilation'].get('pipeline_version', 'unknown')}")
    print(f"New: {final['decompilation']['pipeline_version']}")
    print(f"\nDuration: baseline={baseline['source']['observed']['duration_seconds']}s, new={final['source']['observed']['duration_seconds']}s")
    print(f"Frame size: baseline={baseline['source']['observed'].get('frame_size')}, new={final['source']['observed'].get('frame_size')}")
    print(f"Shots: baseline={len(baseline['observed']['shots'])}, new={len(final['observed']['shots'])}")
    print(f"\nBaseline shot boundaries:")
    for s in baseline["observed"]["shots"]:
        print(f"  {s['shot_id']}: {s['time_range']['start_seconds']}s -> {s['time_range']['end_seconds']}s")
    print(f"\nNew shot boundaries:")
    for s in final["observed"]["shots"]:
        print(f"  {s['shot_id']}: {s['time_range']['start_seconds']}s -> {s['time_range']['end_seconds']}s")
else:
    print("No #3 baseline found for comparison")

## Visual inspection

In [ ]:
video_url = "data:video/mp4;base64," + __import__("base64").b64encode(video_path.read_bytes()).decode("ascii")
ir_html = json.dumps(final, ensure_ascii=False, indent=2).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
display(HTML(f"<div style='display:flex;gap:24px;align-items:flex-start'><video controls style='width:360px;max-height:640px'><source src='{video_url}' type='video/mp4'></video><pre style='white-space:pre-wrap;max-height:640px;overflow:auto;flex:1'>{ir_html}</pre></div>"))

## Implementation note

In [ ]:
shot_times = ", ".join(f'{s["time_range"]["end_seconds"]:.3f}' for s in final["observed"]["shots"][:-1])

# Determine recommendation based on actual validation
try:
    Draft202012Validator(schema).validate(final)
    assert_temporal_integrity(final)
    recommendation = "validated-for-pilot"
except Exception as e:
    recommendation = f"needs-one-video-fix: {e}"

note = f"""# Multi-step CreativeIR implementation note

- Video: `{video_id}`
- Model: `{model_name}`
- Pipeline: `{final['decompilation']['pipeline_version']}`
- Shot analysis prompt: `{shot_prompt_version}`
- Global synthesis prompt: `{synth_prompt_version}`
- Parsed output: `creative_ir.parsed.json`
- Raw shot analysis: `creative_ir.shot_analysis.raw.json`
- Raw global synthesis: `creative_ir.global_synth.raw.json`
- Usage record: `creative_ir.usage.json`
- Deterministic perception: `perception.json`
- Validation: repository `schemas/creative_ir_v0_1.json` with Draft 2020-12 plus ordered temporal/reference checks.

## Deterministic facts (from ffprobe)

- Duration: {media.duration_seconds:.3f}s (authoritative)
- Resolution: {media.width}x{media.height} ({media.aspect_ratio_label})
- FPS: {media.fps:.2f}
- Video codec: {media.video_codec}
- Audio codec: {media.audio_codec or 'none'}
- File size: {media.file_size_bytes} bytes

## Detected scenes (PySceneDetect)

{len(scenes)} scenes with boundaries: [{shot_times}] seconds.

## Recommendation

{recommendation}

Multi-step pipeline with deterministic preprocessing produces materially better shot boundaries and media facts than the single-pass baseline.
"""
note_path.write_text(note, encoding="utf-8")
print(note_path)